# Dual-SR830 frequency and current–voltage sweep browser

Read-only analysis for standalone frequency, excitation, frequency×excitation, and temperature×excitation records. Standalone scans render separate Vxx/Vxy h1/h2/h3 figures. Temperature×excitation records render separate amplitude and phase figures for each available role and harmonic, with one curve per actual formal-window temperature. Missing data is labeled explicitly; no values are inferred.

In [ ]:
import json
import math
import sys
from dataclasses import asdict
from pathlib import Path

working_directory = Path.cwd().resolve()
PROJECT_ROOT = (
    working_directory.parent
    if working_directory.name.lower() == 'notebooks'
    else working_directory
)
SOURCE_DIRECTORY = PROJECT_ROOT / 'src'
if not (SOURCE_DIRECTORY / 'attodry_control').is_dir():
    raise RuntimeError(
        f'Cannot find the project source directory: {SOURCE_DIRECTORY}'
    )
source_directory_text = str(SOURCE_DIRECTORY)
if source_directory_text not in sys.path:
    sys.path.insert(0, source_directory_text)

import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

from attodry_control.commissioning_analysis import (
    ExcitationPathResistance,
    HarmonicScalingRules,
    discover_commissioning_records,
    excitation_path_from_sweep_files,
    export_commissioning_csv,
    load_sweep_sample_files,
    fit_harmonic_scalings,
    plot_harmonic_scaling_fit,
    plot_multi_frequency_iv_curves,
    plot_six_role_harmonic_sweeps,
)
from attodry_control.temperature_excitation_analysis import (
    discover_temperature_excitation_records,
    export_temperature_excitation_csv,
    load_temperature_excitation_sample_files,
    plot_temperature_iv_suite,
)

DATA_DIRECTORY = PROJECT_ROOT / 'run_data' / 'commissioning'
TEMPERATURE_DATA_DIRECTORY = (
    PROJECT_ROOT / 'run_data' / 'temperature_excitation_commissioning'
)

# Analysis-only thresholds. Edit these values for the current data-quality
# decision; they never change the hardware sweep or safety protocol.
SCALING_RULES = HarmonicScalingRules(
    confidence_level=0.95,
    minimum_points=6,
    minimum_current_decades=1.0,
    minimum_snr=3.0,
    max_exponent_ci_width=0.5,
    max_delta_aicc_consistent=2.0,
    min_delta_aicc_inconsistent=6.0,
    max_relative_rmse=0.10,
    max_phase_slope_deg_per_decade=5.0,
    max_phase_span_deg=10.0,
    # Scalar R background: 'auto', 'none', or 'with_offset'.
    scalar_background_mode='auto',
    # Complex Z background: 'auto', 'none', or 'with_offset'.
    complex_background_mode='auto',
    complex_free_exponent_min=0.05,
    complex_free_exponent_max=6.0,
)

# Configure data directories above; plotting cells display figures only.

## Temperature-stacked current–Vxx/Vxy and phase curves

This section reads completed temperature conditions from temperature–excitation summary JSON or formal CSV files. Set `TEMPERATURE_DATA_DIRECTORY` in the first cell, refresh the catalog, select one or more records, then load them. Select temperature conditions and an optional archived RMS-current interval; figures use only their intersection. The current coordinate is the recorded readback-derived `nominal_current_a_rms`; it is never recalculated from today's TOML. Each available Vxx/Vxy × h1/h2/h3 channel produces one amplitude figure and one phase figure, with one colored curve per selected condition. Legend temperatures are actual formal-window means, not requested setpoints, and are drawn outside the plot frame on the right.

Phase repeats use circular mean and circular standard deviation, then each temperature curve is unwrapped only along increasing current for display. No phase setting or raw value is changed. The default sample filter is `clean`; selecting another status is an explicit audit view.

In [ ]:
temperature_sample_status_widget = widgets.SelectMultiple(
    options=('clean', 'problem', 'unlocked', 'overload', 'instrument_error'),
    value=('clean',),
    description='T formal status',
    layout=widgets.Layout(width='95%', height='100px'),
)
temperature_excitation_record_widget = widgets.SelectMultiple(
    options=(),
    description='T × excitation',
    layout=widgets.Layout(width='95%', height='150px'),
)
temperature_condition_widget = widgets.SelectMultiple(
    options=(),
    description='Temperatures',
    layout=widgets.Layout(width='95%', height='170px'),
)
temperature_current_minimum_widget = widgets.Text(
    value='',
    placeholder='blank = no lower limit',
    description='I min (A RMS)',
    layout=widgets.Layout(width='47%'),
)
temperature_current_maximum_widget = widgets.Text(
    value='',
    placeholder='blank = no upper limit',
    description='I max (A RMS)',
    layout=widgets.Layout(width='47%'),
)
refresh_temperature_records_button = widgets.Button(
    description='Refresh T records', icon='refresh', button_style='info'
)
load_temperature_records_button = widgets.Button(
    description='Load T records', icon='check', button_style='success'
)
apply_temperature_filter_button = widgets.Button(
    description='Apply T × I filters', icon='filter', button_style='warning'
)
temperature_selector_message = widgets.HTML()

TEMPERATURE_EXCITATION_PATHS = ()
temperature_excitation_paths = ()
temperature_excitation_loaded_rows = ()
temperature_excitation_rows = ()
temperature_current_minimum_a_rms = None
temperature_current_maximum_a_rms = None
temperature_iv_figures = {}

def _temperature_condition_key(row):
    return f'{row.source_path}::{row.temperature_index}'

def _temperature_condition_options(rows):
    grouped = {}
    for row in rows:
        grouped.setdefault(_temperature_condition_key(row), row)
    return tuple(
        (
            f'{row.condition_measurement_temperature_k:.6g} K measured | '
            f'{row.requested_temperature_k:.6g} K set | '
            f'{Path(row.source_path).name} | T#{row.temperature_index}',
            key,
        )
        for key, row in sorted(
            grouped.items(),
            key=lambda item: (
                item[1].condition_measurement_temperature_k,
                item[1].source_path,
                item[1].temperature_index,
            ),
        )
    )

def _parse_optional_current_bound(value, label):
    text = value.strip()
    if not text:
        return None
    try:
        current = float(text)
    except ValueError as exc:
        raise ValueError(f'{label} must be a number in A RMS or blank.') from exc
    if not math.isfinite(current) or current < 0.0:
        raise ValueError(f'{label} must be a finite non-negative current.')
    return current

def _refresh_temperature_records(_=None):
    records = discover_temperature_excitation_records(
        TEMPERATURE_DATA_DIRECTORY
    )
    temperature_excitation_record_widget.options = tuple(
        (path.name, str(path)) for path in records
    )
    temperature_selector_message.value = (
        f'<b>Directory:</b> {TEMPERATURE_DATA_DIRECTORY}<br>'
        f'Found {len(records)} temperature–excitation records.'
    )

def _load_temperature_records(_=None):
    global TEMPERATURE_EXCITATION_PATHS
    global temperature_excitation_paths, temperature_excitation_loaded_rows
    global temperature_excitation_rows
    TEMPERATURE_EXCITATION_PATHS = tuple(
        Path(value) for value in temperature_excitation_record_widget.value
    )
    if not TEMPERATURE_EXCITATION_PATHS:
        temperature_selector_message.value = '<b>No temperature–excitation record selected.</b>'
        return
    try:
        temperature_excitation_loaded_rows = load_temperature_excitation_sample_files(
            TEMPERATURE_EXCITATION_PATHS,
            sample_statuses=set(temperature_sample_status_widget.value),
        )
    except ValueError as exc:
        temperature_selector_message.value = f'<b>Could not load temperature data:</b> {exc}'
        return
    temperature_excitation_paths = TEMPERATURE_EXCITATION_PATHS
    temperature_condition_widget.options = _temperature_condition_options(
        temperature_excitation_loaded_rows
    )
    temperature_condition_widget.value = tuple(
        value for _label, value in temperature_condition_widget.options
    )
    temperature_excitation_rows = temperature_excitation_loaded_rows
    temperature_selector_message.value = (
        f'<b>Loaded:</b> {len(temperature_excitation_paths)} file(s), '
        f'{len(temperature_excitation_rows)} selected formal rows, '
        f'{len(temperature_condition_widget.options)} temperature conditions.'
    )

def _apply_temperature_filter(_=None):
    global temperature_excitation_rows
    global temperature_current_minimum_a_rms
    global temperature_current_maximum_a_rms
    selected = set(temperature_condition_widget.value)
    try:
        minimum_current = _parse_optional_current_bound(
            temperature_current_minimum_widget.value, 'I min'
        )
        maximum_current = _parse_optional_current_bound(
            temperature_current_maximum_widget.value, 'I max'
        )
    except ValueError as exc:
        temperature_selector_message.value = f'<b>Current filter error:</b> {exc}'
        return
    if (
        minimum_current is not None
        and maximum_current is not None
        and minimum_current > maximum_current
    ):
        temperature_selector_message.value = '<b>Current filter error:</b> I min cannot exceed I max.'
        return
    temperature_excitation_rows = tuple(
        row
        for row in temperature_excitation_loaded_rows
        if (
            _temperature_condition_key(row) in selected
            and (minimum_current is None or row.current_a_rms >= minimum_current)
            and (maximum_current is None or row.current_a_rms <= maximum_current)
        )
    )
    temperature_current_minimum_a_rms = minimum_current
    temperature_current_maximum_a_rms = maximum_current
    minimum_label = 'no lower limit' if minimum_current is None else f'{minimum_current:.6g}'
    maximum_label = 'no upper limit' if maximum_current is None else f'{maximum_current:.6g}'
    temperature_selector_message.value = (
        f'<b>T × I selection applied:</b> {len(selected)} condition(s), '
        f'I = {minimum_label} to {maximum_label} A RMS, '
        f'{len(temperature_excitation_rows)} formal rows. Run the figure cell.'
    )

refresh_temperature_records_button.on_click(_refresh_temperature_records)
load_temperature_records_button.on_click(_load_temperature_records)
apply_temperature_filter_button.on_click(_apply_temperature_filter)
_refresh_temperature_records()
display(widgets.VBox([
    widgets.HBox([refresh_temperature_records_button, load_temperature_records_button]),
    temperature_sample_status_widget,
    temperature_excitation_record_widget,
    temperature_condition_widget,
    widgets.HBox([
        temperature_current_minimum_widget,
        temperature_current_maximum_widget,
    ]),
    apply_temperature_filter_button,
    temperature_selector_message,
]))

In [ ]:
temperature_iv_figures = (
    plot_temperature_iv_suite(temperature_excitation_rows)
    if temperature_excitation_rows
    else {}
)
for figure in temperature_iv_figures.values():
    display(figure)
    plt.close(figure)

## Start here: filters, remote record selection, and current calibration

Set `DATA_DIRECTORY` once in the preceding cell. Click **Refresh records**, select either or both record types, then click **Load selected records**. A frequency record produces the six frequency figures; an excitation record produces the six current-voltage figures. The `completed` checkbox is on by default; deselect it only for an explicit audit. `clean` formal samples are the default automatic quality screen. Current is always `SINE OUT RMS voltage / (external series + SR830 output + approximate device resistance)`, using the path archived with each sweep from `hardware.local.toml`. Do not duplicate normal resistance values here.

### Harmonic scaling decision rules

The `SCALING_RULES` block in the first code cell is intentionally editable. It is analysis-only and is copied into `selection_manifest.json` when outputs are saved. For each available XX/XY and h1/h2/h3 excitation channel, the analysis compares three distinct views. The log-magnitude fit is `log R = log A + p log I` against fixed `p=n`. The phase-blind scalar fit is `R = b + A·(I/Iref)^p` against fixed `p=n`; it uses only measured amplitude R, ignores phase and X/Y, and constrains b and A to be non-negative. `scalar_phase_ignored=True` is retained in every exported result so this choice is explicit. The complex fit is performed directly on X/Y.

`minimum_points` is the minimum number of current points; `minimum_current_decades` is the required log10(Imax/Imin) span; `minimum_snr` excludes a point only when replicate standard error is available and the estimated SNR is below the threshold. `max_exponent_ci_width` limits the width of the approximate confidence interval for the fitted exponent. `max_delta_aicc_consistent` and `min_delta_aicc_inconsistent` compare fixed-order and free-exponent models, where ΔAICc = AICc_fixed − AICc_free. `max_relative_rmse` limits the fixed-order relative error. Set optional thresholds to `None` to disable that criterion.

For the scalar fit, `scalar_background_mode='auto'` uses corrected AIC to choose between no-offset and offset fixed-order models; `'none'` or `'with_offset'` forces one. The four scalar models and `scalar_R_verdict` are exported. Their AICc values are comparable within the scalar method, but should not be compared directly with log-space or complex-space AICc because those methods use different residual spaces.
The complex, background-aware comparison works on `Z=X+iY`, not on magnitude alone. It calculates four models: `Z=C·I^n`, `Z=C·I^p`, `Z=B+C·I^n`, and `Z=B+C·I^p`. Here `B` is a complex background with its own amplitude and phase. `complex_background_mode='auto'` uses corrected AIC to choose whether the fixed-order comparison uses `B`; `'none'` forces no background and `'with_offset'` forces it. `complex_power_law_verdict` is the offset-aware conclusion; its `complex_models` export contains the fitted `B`, response vector at the geometric-mean current reference, AICc, residual, exponent, and confidence interval. Raw phase can rotate because `B` and `C·I^n` have different phases, so `complex_response_verdict` remains a separate raw-phase quality check. `R²` is contextual only and never the sole decision rule.

In [ ]:
completed_only_widget = widgets.Checkbox(
    value=True,
    description='Only completed records',
)
sample_status_widget = widgets.SelectMultiple(
    options=('clean', 'problem', 'unlocked', 'overload', 'instrument_error'),
    value=('clean',),
    description='Formal samples',
)
include_rejected_widget = widgets.Checkbox(
    value=False,
    description='Allow rejected audit records',
)
refresh_records_button = widgets.Button(
    description='Refresh records',
    icon='refresh',
    button_style='info',
)
frequency_record_widget = widgets.Dropdown(
    options=(),
    description='Frequency',
    layout=widgets.Layout(width='95%'),
)
excitation_record_widget = widgets.Dropdown(
    options=(),
    description='Excitation',
    layout=widgets.Layout(width='95%'),
)
combined_record_widget = widgets.Dropdown(
    options=(),
    description='Freq × amp',
    layout=widgets.Layout(width='95%'),
)
load_selected_records_button = widgets.Button(
    description='Load selected records',
    icon='check',
    button_style='success',
)
apply_point_exclusions_button = widgets.Button(
    description='Apply point exclusions',
    icon='filter',
    button_style='warning',
)
frequency_excluded_points_widget = widgets.SelectMultiple(
    options=(),
    description='Exclude frequency points',
    layout=widgets.Layout(width='95%', height='120px'),
)
excitation_excluded_points_widget = widgets.SelectMultiple(
    options=(),
    description='Exclude excitation points',
    layout=widgets.Layout(width='95%', height='120px'),
)
selector_message = widgets.HTML()
point_filter_message = widgets.HTML()

FREQUENCY_PATHS = ()
EXCITATION_PATHS = ()
COMBINED_PATHS = ()
FREQUENCY_EXCLUDED_POINT_KEYS = set()
EXCITATION_EXCLUDED_POINT_KEYS = set()

def _sync_filters():
    global RECORD_STATUSES, SAMPLE_STATUSES, INCLUDE_REJECTED
    RECORD_STATUSES = {'completed'} if completed_only_widget.value else None
    SAMPLE_STATUSES = set(sample_status_widget.value)
    INCLUDE_REJECTED = include_rejected_widget.value

def _record_options(scan_type):
    records = discover_commissioning_records(
        DATA_DIRECTORY,
        record_statuses=RECORD_STATUSES,
        scan_types={scan_type},
    )
    return [
        (
            f'{record.path.name} | {record.record_status} | '
            f'{record.sample_count} formal samples',
            str(record.path),
        )
        for record in records
    ]

def _refresh_records(_=None):
    _sync_filters()
    frequency_options = _record_options('frequency')
    excitation_options = _record_options('excitation')
    combined_options = _record_options('frequency_excitation')
    frequency_record_widget.options = frequency_options
    excitation_record_widget.options = excitation_options
    combined_record_widget.options = combined_options
    selector_message.value = (
        f'<b>Directory:</b> {DATA_DIRECTORY}<br>'
        f'Found {len(frequency_options)} frequency, {len(excitation_options)} excitation, '
        f'and {len(combined_options)} frequency×amplitude records.'
    )

def _load_selected_records(_):
    global FREQUENCY_PATHS, EXCITATION_PATHS, COMBINED_PATHS
    _sync_filters()
    frequency_path = frequency_record_widget.value
    excitation_path = excitation_record_widget.value
    combined_path = combined_record_widget.value
    FREQUENCY_PATHS = (Path(frequency_path),) if frequency_path else ()
    EXCITATION_PATHS = (Path(excitation_path),) if excitation_path else ()
    COMBINED_PATHS = (Path(combined_path),) if combined_path else ()
    selected_types = []
    if FREQUENCY_PATHS:
        selected_types.append('frequency: ' + FREQUENCY_PATHS[0].name)
    if EXCITATION_PATHS:
        selected_types.append('excitation: ' + EXCITATION_PATHS[0].name)
    if COMBINED_PATHS:
        selected_types.append('frequency×amplitude: ' + COMBINED_PATHS[0].name)
    if not selected_types:
        selector_message.value = (
            '<b>No record selected.</b> Choose at least one frequency, excitation, '
            'or frequency×amplitude record, then load it.'
        )
        return
    try:
        loaded = _load_selected_formal_samples()
    except ValueError as exc:
        selector_message.value = f'<b>Could not load selected samples:</b> {exc}'
        return
    selector_message.value = (
        '<b>Loaded:</b> ' + '; '.join(selected_types) + '<br>'
        f"Points ready: {loaded['frequency_selected_rows']} frequency rows; "
        f"{loaded['excitation_selected_rows']} excitation rows; "
        f"{loaded['combined_selected_rows']} combined rows. Choose any "
        'points to exclude, then run the figure cell.'
    )

def _point_key(row):
    return f'{row.source_path}::{row.point_index}'

def _point_options(rows, scan_type, excitation_path):
    grouped = {}
    for row in rows:
        grouped.setdefault(_point_key(row), []).append(row)
    options = []
    for key, point_rows in grouped.items():
        first = point_rows[0]
        coordinate = (
            f'{first.actual_frequency_hz:g} Hz'
            if scan_type == 'frequency'
            else f'{excitation_path.current_from_sine_output(first.sine_output_v_rms):.4g} A RMS'
        )
        channels = ', '.join(
            f'{role} h{harmonic}'
            for role, harmonic in sorted({(row.role, row.harmonic) for row in point_rows})
        )
        label = (
            f'#{first.point_index} | {coordinate} | {len(point_rows)} selected rows | {channels}'
        )
        options.append((label, key))
    return tuple(options)

def _load_selected_formal_samples():
    _sync_filters()
    global FREQUENCY_EXCLUDED_POINT_KEYS, EXCITATION_EXCLUDED_POINT_KEYS
    global frequency_paths, excitation_paths, combined_paths
    global frequency_loaded_rows, excitation_loaded_rows
    global frequency_excitation_path, excitation_excitation_path, combined_excitation_path
    global frequency_rows, excitation_rows, combined_rows
    frequency_paths = tuple(FREQUENCY_PATHS)
    excitation_paths = tuple(EXCITATION_PATHS)
    combined_paths = tuple(COMBINED_PATHS)
    frequency_loaded_rows = (
        load_sweep_sample_files(
            frequency_paths,
            include_rejected=INCLUDE_REJECTED,
            sample_statuses=SAMPLE_STATUSES,
        )
        if frequency_paths
        else ()
    )
    excitation_loaded_rows = (
        load_sweep_sample_files(
            excitation_paths,
            include_rejected=INCLUDE_REJECTED,
            sample_statuses=SAMPLE_STATUSES,
        )
        if excitation_paths
        else ()
    )
    combined_loaded_rows = (
        load_sweep_sample_files(
            combined_paths,
            include_rejected=INCLUDE_REJECTED,
            sample_statuses=SAMPLE_STATUSES,
        )
        if combined_paths
        else ()
    )
    frequency_excitation_path = (
        excitation_path_from_sweep_files(
            frequency_paths,
            excitation_path_override=EXCITATION_PATH_OVERRIDE,
        )
        if frequency_paths
        else None
    )
    excitation_excitation_path = (
        excitation_path_from_sweep_files(
            excitation_paths,
            excitation_path_override=EXCITATION_PATH_OVERRIDE,
        )
        if excitation_paths
        else None
    )
    combined_excitation_path = (
        excitation_path_from_sweep_files(
            combined_paths,
            excitation_path_override=EXCITATION_PATH_OVERRIDE,
        )
        if combined_paths
        else None
    )
    frequency_excluded_points_widget.options = (
        _point_options(frequency_loaded_rows, 'frequency', frequency_excitation_path)
        if frequency_loaded_rows
        else ()
    )
    excitation_excluded_points_widget.options = (
        _point_options(excitation_loaded_rows, 'excitation', excitation_excitation_path)
        if excitation_loaded_rows
        else ()
    )
    frequency_excluded_points_widget.value = ()
    excitation_excluded_points_widget.value = ()
    FREQUENCY_EXCLUDED_POINT_KEYS = set()
    EXCITATION_EXCLUDED_POINT_KEYS = set()
    frequency_rows = frequency_loaded_rows
    excitation_rows = excitation_loaded_rows
    combined_rows = combined_loaded_rows
    point_filter_message.value = (
        '<b>Automatic quality screen:</b> ' + ', '.join(sorted(SAMPLE_STATUSES)) +
        '. Select any suspect scan points above, apply exclusions, then run the figure cell.'
    )
    return {
        'frequency_files': frequency_paths,
        'frequency_selected_rows': len(frequency_rows),
        'frequency_total_path_resistance_ohm': (
            frequency_excitation_path.total_resistance_ohm
            if frequency_excitation_path is not None
            else None
        ),
        'excitation_files': excitation_paths,
        'excitation_selected_rows': len(excitation_rows),
        'excitation_total_path_resistance_ohm': (
            excitation_excitation_path.total_resistance_ohm
            if excitation_excitation_path is not None
            else None
        ),
        'combined_files': combined_paths,
        'combined_selected_rows': len(combined_rows),
        'combined_total_path_resistance_ohm': (
            combined_excitation_path.total_resistance_ohm
            if combined_excitation_path is not None
            else None
        ),
    }

def _rows_after_point_exclusions(rows, excluded_keys):
    return tuple(row for row in rows if _point_key(row) not in excluded_keys)

def _apply_point_exclusions(_):
    global FREQUENCY_EXCLUDED_POINT_KEYS, EXCITATION_EXCLUDED_POINT_KEYS
    global frequency_rows, excitation_rows, combined_rows
    FREQUENCY_EXCLUDED_POINT_KEYS = set(frequency_excluded_points_widget.value)
    EXCITATION_EXCLUDED_POINT_KEYS = set(excitation_excluded_points_widget.value)
    frequency_rows = _rows_after_point_exclusions(
        globals().get('frequency_loaded_rows', ()), FREQUENCY_EXCLUDED_POINT_KEYS
    )
    excitation_rows = _rows_after_point_exclusions(
        globals().get('excitation_loaded_rows', ()), EXCITATION_EXCLUDED_POINT_KEYS
    )
    point_filter_message.value = (
        f'<b>Plot selection applied.</b> Frequency: {len(frequency_rows)} rows '
        f'after excluding {len(FREQUENCY_EXCLUDED_POINT_KEYS)} points; '
        f'excitation: {len(excitation_rows)} rows; combined: {len(combined_rows)} rows after excluding '
        f'{len(EXCITATION_EXCLUDED_POINT_KEYS)} points. Run the figure cell.'
    )

refresh_records_button.on_click(_refresh_records)
load_selected_records_button.on_click(_load_selected_records)
apply_point_exclusions_button.on_click(_apply_point_exclusions)
_refresh_records()
display(widgets.VBox([
    widgets.HBox([refresh_records_button, load_selected_records_button, completed_only_widget, include_rejected_widget]),
    sample_status_widget,
    frequency_record_widget,
    excitation_record_widget,
    combined_record_widget,
    selector_message,
    frequency_excluded_points_widget,
    excitation_excluded_points_widget,
    apply_point_exclusions_button,
    point_filter_message,
]))

_sync_filters()

# Default: use measurement_config.excitation_path recorded in each selected JSON.
# Set this only for legacy JSON that lacks that snapshot; it deliberately
# overrides every selected file, so do not use it for normal daily records.
EXCITATION_PATH_OVERRIDE: ExcitationPathResistance | None = None
# EXCITATION_PATH_OVERRIDE = ExcitationPathResistance(
#     external_series_resistance_ohm=...,
#     sr830_output_resistance_ohm=...,
#     approximate_device_resistance_ohm=...,
# )
# Plot phase only above this amplitude and below this circular sample spread.
# Set the amplitude to 0.0 and the spread to None to inspect all raw phases.
PHASE_MINIMUM_AMPLITUDE_V = 1e-6
PHASE_MAXIMUM_STANDARD_DEVIATION_DEG = 5.0
# The selection UI above sets these tuples. The notebook never silently
# substitutes a newer record when no record of that type was selected.

# Plot settings above take effect when the figure cell is run.

## Filtered catalog

The catalog is newest-first. Toggle the completed checkbox, choose formal-sample statuses, then click **Refresh records**. Select either scan type independently; the notebook does not silently substitute a newer record.

In [ ]:
RECORD_STATUSES = {'completed'} if completed_only_widget.value else None
SAMPLE_STATUSES = set(sample_status_widget.value)
INCLUDE_REJECTED = include_rejected_widget.value

# The controls above refresh and show the filtered record catalog.

## Load selected formal samples

The loader excludes transition and cleanup payloads. It refuses rejected records unless `INCLUDE_REJECTED=True`. Frequency and excitation records stay separate and either can be empty. The `clean` status is the default automatic quality screen. Clicking **Load selected records** already fills the point selectors above; rerun this cell only after changing the formal-sample filter. Select suspect points, click **Apply point exclusions**, then run the figure cell. Clearing the selections and applying again restores all automatically retained points.

In [ ]:
# This remains available for rerunning after changing a formal-sample filter.
# The Load selected records button already calls it for the normal workflow.
_ = _load_selected_formal_samples()

## Available frequency and current–voltage figures

A loaded frequency record produces six frequency figures with a logarithmic frequency axis and SINE OUT-derived RMS current in their titles. A loaded excitation record produces six current-voltage figures using the same calculated current. Each figure keeps voltage magnitude and phase on separate y axes. Missing scan types are skipped; missing harmonics are labeled explicitly.

For excitation data, the next part of the cell performs the editable harmonic-scaling analysis. It fits every available XX/XY and h1/h2/h3 combination independently and plots the log-space fits, the selected phase-blind scalar-R curve, and the selected complex-background curve. The Notebook displays figures only: legends are outside the axes on the right and numerical fit results are retained in the optional export manifest rather than rendered in the Notebook. Use `scalar_R_verdict` when phase is too unstable to trust; it is explicitly an amplitude-only result. Use `complex_power_law_verdict` for the background-aware X/Y result; `complex_response_verdict` remains a separate raw-phase stability audit.

In [ ]:
frequency_figures = {}
current_voltage_figures = {}
combined_iv_figures = {}
harmonic_scaling_results = {}
harmonic_scaling_figures = {}
if frequency_rows:
    frequency_figures = plot_six_role_harmonic_sweeps(
        frequency_rows,
        excitation_path=frequency_excitation_path,
        phase_minimum_amplitude_v=PHASE_MINIMUM_AMPLITUDE_V,
        phase_maximum_standard_deviation_deg=PHASE_MAXIMUM_STANDARD_DEVIATION_DEG,
    )
if excitation_rows:
    current_voltage_figures = plot_six_role_harmonic_sweeps(
        excitation_rows,
        excitation_path=excitation_excitation_path,
        phase_minimum_amplitude_v=PHASE_MINIMUM_AMPLITUDE_V,
        phase_maximum_standard_deviation_deg=PHASE_MAXIMUM_STANDARD_DEVIATION_DEG,
    )
if combined_rows:
    combined_iv_figures = {
        (role, harmonic): plot_multi_frequency_iv_curves(
            combined_rows,
            role=role,
            harmonic=harmonic,
            metric='amplitude_v',
            excitation_path=combined_excitation_path,
        )
        for role in ('xx', 'xy')
        for harmonic in (1, 2, 3)
        if any(row.role == role and row.harmonic == harmonic for row in combined_rows)
    }

for _scan_name, figures in (
    ('frequency', frequency_figures),
    ('current_voltage', current_voltage_figures),
    ('combined_current_voltage', combined_iv_figures),
):
    for figure in figures.values():
        display(figure)
        plt.close(figure)

if excitation_rows:
    harmonic_scaling_results = fit_harmonic_scalings(
        excitation_rows,
        excitation_path=excitation_excitation_path,
        rules=SCALING_RULES,
    )
    harmonic_scaling_figures = {
        key: plot_harmonic_scaling_fit(result)
        for key, result in harmonic_scaling_results.items()
    }
    for figure in harmonic_scaling_figures.values():
        display(figure)
        plt.close(figure)

## Optional export

No files are written unless `SAVE_OUTPUTS=True`. Exports are placed under the ignored analysis-output directory and include a JSON selection manifest with filters, chosen files, excluded scan points, selected temperature conditions, the editable scaling rules, and each fit result.

In [ ]:
SAVE_OUTPUTS = False
OUTPUT_DIRECTORY = PROJECT_ROOT / 'analysis_output' / 'sr830_commissioning'
if SAVE_OUTPUTS:
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    if frequency_rows:
        export_commissioning_csv(
            frequency_rows, OUTPUT_DIRECTORY / 'frequency_samples.csv'
        )
    if excitation_rows:
        export_commissioning_csv(
            excitation_rows, OUTPUT_DIRECTORY / 'excitation_samples.csv'
        )
    if combined_rows:
        export_commissioning_csv(
            combined_rows, OUTPUT_DIRECTORY / 'frequency_excitation_samples.csv'
        )
    if temperature_excitation_rows:
        export_temperature_excitation_csv(
            temperature_excitation_rows,
            OUTPUT_DIRECTORY / 'temperature_excitation_samples.csv',
        )
    for scan_name, figures in (
        ('frequency', frequency_figures),
        ('current_voltage', current_voltage_figures),
        ('combined_current_voltage', combined_iv_figures),
    ):
        for (role, harmonic), figure in figures.items():
            stem = f'{scan_name}_{role}_h{harmonic}'
            figure.savefig(OUTPUT_DIRECTORY / f'{stem}.png', dpi=200)
            figure.savefig(OUTPUT_DIRECTORY / f'{stem}.pdf')
    for (role, harmonic, metric), figure in temperature_iv_figures.items():
        stem = f'temperature_iv_{role}_h{harmonic}_{metric}'
        figure.savefig(OUTPUT_DIRECTORY / f'{stem}.png', dpi=200)
        figure.savefig(OUTPUT_DIRECTORY / f'{stem}.pdf')
    for (role, harmonic), figure in harmonic_scaling_figures.items():
        stem = f'harmonic_scaling_{role}_h{harmonic}'
        figure.savefig(OUTPUT_DIRECTORY / f'{stem}.png', dpi=200)
        figure.savefig(OUTPUT_DIRECTORY / f'{stem}.pdf')
    selection_manifest = {
        'data_directory': str(DATA_DIRECTORY),
        'temperature_data_directory': str(TEMPERATURE_DATA_DIRECTORY),
        'filters': {
            'record_statuses': sorted(RECORD_STATUSES) if RECORD_STATUSES else None,
            'sample_statuses': sorted(SAMPLE_STATUSES),
            'include_rejected': INCLUDE_REJECTED,
        },
        'phase_display': {
            'PHASE_MINIMUM_AMPLITUDE_V': PHASE_MINIMUM_AMPLITUDE_V,
            'PHASE_MAXIMUM_STANDARD_DEVIATION_DEG': PHASE_MAXIMUM_STANDARD_DEVIATION_DEG,
        },
        'harmonic_scaling_rules': asdict(SCALING_RULES),
        'harmonic_scaling_results': {
            f'{role}_h{harmonic}': result.as_dict()
            for (role, harmonic), result in harmonic_scaling_results.items()
        },
        'frequency': {
            'files': [str(path) for path in frequency_paths],
            'excluded_point_keys': sorted(FREQUENCY_EXCLUDED_POINT_KEYS),
            'selected_rows': len(frequency_rows),
        },
        'excitation': {
            'files': [str(path) for path in excitation_paths],
            'excluded_point_keys': sorted(EXCITATION_EXCLUDED_POINT_KEYS),
            'selected_rows': len(excitation_rows),
        },
        'frequency_excitation': {
            'files': [str(path) for path in combined_paths],
            'selected_rows': len(combined_rows),
        },
        'temperature_excitation': {
            'files': [str(path) for path in temperature_excitation_paths],
            'sample_statuses': sorted(temperature_sample_status_widget.value),
            'selected_condition_keys': sorted(temperature_condition_widget.value),
            'current_minimum_a_rms': temperature_current_minimum_a_rms,
            'current_maximum_a_rms': temperature_current_maximum_a_rms,
            'selected_rows': len(temperature_excitation_rows),
            'phase_statistics': 'circular mean/std; unwrapped along current for display',
        },
    }
    (OUTPUT_DIRECTORY / 'selection_manifest.json').write_text(
        json.dumps(selection_manifest, indent=2), encoding='utf-8'
    )
    display(OUTPUT_DIRECTORY)